In [4]:
.root <- getwd()
while (!file.exists(file.path(.root, 'CLAUDE.md')) && dirname(.root) != .root)
  .root <- dirname(.root)
setwd(.root)
rm(.root)

suppressPackageStartupMessages({
  library(tidyverse)
  library(arrow)
  library(ggthemes)
  library(scales)
  library(patchwork)
})

options(repr.plot.width = 14, repr.plot.height = 8)

BASE_THEME <- theme_fivethirtyeight() +
  theme(
    axis.title        = element_text(size = 14),
    axis.text         = element_text(size = 13),
    panel.background  = element_rect(fill = 'white', color = NA),
    plot.background   = element_rect(fill = 'white', color = NA),
    legend.background = element_rect(fill = 'white', color = NA),
    legend.text       = element_text(size = 12),
    legend.title      = element_text(size = 13),
    strip.text        = element_text(size = 14, face = 'bold'),
    plot.title        = element_text(size = 20, face = 'bold'),
    plot.subtitle     = element_text(size = 13)
  )
theme_set(BASE_THEME)

dir.create('results/plots/results', recursive = TRUE, showWarnings = FALSE)

In [5]:
xrv <- read_parquet('results/xrv_causal.parquet') |>
  select(disruption_tax, distortion_tax, selection_tax,
         spatial_distortion_tax, distortion_share) |>
  filter(!is.na(distortion_tax), !is.na(selection_tax), !is.na(distortion_share))

cat(sprintf(
  'Loaded %s swings | median distortion_share = %.3f | mean distortion_share = %.3f\n',
  formatC(nrow(xrv), format = 'd', big.mark = ','),
  median(xrv$distortion_share, na.rm = TRUE),
  mean(xrv$distortion_share, na.rm = TRUE)
))

Loaded 761,169 swings | median distortion_share = 0.044 | mean distortion_share = 0.219


## Disruption Attribution: Selection vs Distortion

In [ ]:
mean_share <- mean(xrv$distortion_share)
med_share  <- median(xrv$distortion_share)

p_share <- ggplot(xrv, aes(x = distortion_share)) +
  geom_histogram(fill = '#3498db', color = 'white', linewidth = 0.2,
                 binwidth = 0.025, boundary = 0) +
  geom_vline(xintercept = mean_share, color = '#e74c3c', linewidth = 1.2) +
  annotate('text', x = mean_share + 0.02, y = Inf, vjust = 1.5,
           label = sprintf('Mean = %.0f%%', mean_share * 100),
           color = '#e74c3c', size = 4.5, fontface = 'bold', hjust = 0) +
  scale_x_continuous(labels = percent_format(accuracy = 1),
                     expand = expansion(mult = c(0.01, 0.01)),
                     limits = c(0, 1)) +
  scale_y_continuous(labels = comma_format()) +
  labs(
    title    = 'Disruption Attribution: Selection vs Distortion',
    subtitle = sprintf('Per-swing distortion share  |  %s swings  |  Median = %.0f%%',
                       formatC(nrow(xrv), format = 'd', big.mark = ','),
                       med_share * 100),
    x = 'Distortion share (% of disruption tax from post-commit movement)',
    y = 'Number of swings'
  )

options(repr.plot.width = 14, repr.plot.height = 7)
p_share
ggsave('results/plots/results/distortion_share.png', p_share,
       width = 14, height = 7, dpi = 150)